# Evidential Semantic Entropy — Google Colab Runner

**Paper:** *Evidential Semantic Entropy for LLM Uncertainty Quantification* (EACL 2026 submission)

**Pipeline:**
1. Setup: clone repo, install dependencies
2. Auth: HuggingFace + wandb
3. Run: generate answers → compute uncertainty → analyze results

> Make sure your runtime has **GPU enabled**: `Runtime → Change runtime type → T4 GPU`

## 1. Check GPU

In [2]:
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Mon May 11 14:17:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Clone Repository

In [3]:
import os

REPO_URL = "https://github.com/AfrarJahin/evedential-semantic-entropy.git"  # ← update this
REPO_DIR = "/content/evedential-semantic-entropy"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repo already cloned.")

%cd {REPO_DIR}
!ls

Repo already cloned.
/content/evedential-semantic-entropy
analyze_run.py	   environment_local.yml  __init__.py	requirements.txt
colab_run.ipynb    environment.yml	  kle		semantic_uncertainty
compute_scores.py  evsme		  kle.egg-info	setup.py
configs		   EXP			  README.md	summarize_results.py


In [5]:
import os

REPO_URL = "https://github.com/AfrarJahin/evedential-semantic-entropy.git"
REPO_DIR = "/content/evedential-semantic-entropy"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
    print("Repo updated.")

%cd {REPO_DIR}
!ls


Already up to date.
Repo updated.
/content/evedential-semantic-entropy
analyze_run.py	   environment_local.yml  __init__.py	requirements.txt
colab_run.ipynb    environment.yml	  kle		semantic_uncertainty
compute_scores.py  evsme		  kle.egg-info	setup.py
configs		   EXP			  README.md	summarize_results.py


In [6]:
# Hotfix: patch cloned repo for known issues (applied after clone, before install)
import re, subprocess, sys

# 1. Fix TinyLlama model-ID double-prefix bug in huggingface_models.py
#    Original:  model_id = f'TinyLlama/{model_name}'
#    Problem:   passing 'TinyLlama/TinyLlama-1.1B-Chat-v1.0' makes it triple-slash
#    Fix:       only prepend prefix when the name doesn't already contain '/'
hf_path = '/content/evedential-semantic-entropy/semantic_uncertainty/uncertainty/models/huggingface_models.py'
with open(hf_path) as f:
    code = f.read()

old = "model_id = f'meta-llama/{model_name}'"
new = "model_id = model_name if '/' in model_name else f'meta-llama/{model_name}'"
patched = code.replace(old, new)

with open(hf_path, 'w') as f:
    f.write(patched)
print('[hotfix] TinyLlama model-ID patch:', 'applied' if patched != code else 'already present')

# 2. Install loguru (missing from environment.yml but required by kle/vis_utils.py)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'loguru', '-q'], check=True)
print('[hotfix] loguru installed.')

# 3. Fix wandb retry sleep: 10-minute hang on init failure
#    Original: time.sleep(10 * 60) — blocks for 10 min per failed attempt
#    Fix:      reduce to 5 seconds so failures surface quickly
ga_path = '/content/evedential-semantic-entropy/semantic_uncertainty/generate_answers.py'
with open(ga_path) as f:
    code = f.read()
patched = code.replace('time.sleep(10 * 60)', 'time.sleep(5)')
with open(ga_path, 'w') as f:
    f.write(patched)
print('[hotfix] wandb retry sleep patch:', 'applied' if patched != code else 'already present')


[hotfix] TinyLlama model-ID patch: already present
[hotfix] loguru installed.
[hotfix] wandb retry sleep patch: already present


## 3. Install Dependencies from `environment.yml`

The `environment.yml` is entirely pip-based, so we parse it with PyYAML and feed it straight to `pip install`.

**Packages skipped automatically:**
- `nvidia-*` — Colab already has CUDA drivers; reinstalling these can break them
- `triton` — pre-installed by Colab's PyTorch build
- Jupyter/notebook packages — already present in the Colab runtime
- `-e .` — installed separately in the next cell

In [7]:
import yaml, subprocess, sys

# Packages that must NOT come from the pinned environment.yml list:
#   - torch/torchvision/torchaudio: Colab ships these pre-compiled for its CUDA
#     version; reinstalling from PyPI breaks compiled ops (torchvision::nms etc.)
#   - nvidia-*/triton: Colab's CUDA stack — reinstalling breaks them
#   - Jupyter runtime packages: already managed by Colab
SKIP_PREFIXES = (
    'torch',
    'torchvision',
    'torchaudio',
    'triton',
    'nvidia-',
    'jupyter',
    'notebook',
    'ipython',
    'ipykernel',
    'ipywidgets',
    'widgetsnbextension',
    'jupyterlab',
    'nbformat',
    'nbclient',
    'nbconvert',
    'traitlets',
    'tornado',
    'pyzmq',
    'comm',
    'debugpy',
)

with open('environment.yml') as f:
    env = yaml.safe_load(f)

pip_deps = []
for dep in env.get('dependencies', []):
    if isinstance(dep, dict) and 'pip' in dep:
        pip_deps = dep['pip']
        break

to_install = [
    d for d in pip_deps
    if d != '-e .'
    and not any(d.lower().startswith(p) for p in SKIP_PREFIXES)
]

print(f"Packages to install : {len(to_install)}")
print(f"Skipped             : {len(pip_deps) - len(to_install) - 1}  (torch stack + Colab runtime)")
print("Installing — this may take a few minutes...")

result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q'] + to_install,
    capture_output=True, text=True
)

if result.returncode != 0:
    print("STDERR (last 3000 chars):")
    print(result.stderr[-3000:])
else:
    print("Done.")

Packages to install : 135
Skipped             : 43  (torch stack + Colab runtime)
Installing — this may take a few minutes...
Done.


In [8]:
# Install this repo as an editable package (the '-e .' entry from environment.yml)
!pip install -e . -q
print("Editable install complete.")

  Preparing metadata (setup.py) ... done
Editable install complete.


In [9]:
# Verify imports
import sys
sys.path.insert(0, '/content/evedential-semantic-entropy')
sys.path.insert(0, '/content/evedential-semantic-entropy/semantic_uncertainty')

import torch
import transformers
import wandb
print(f"torch: {torch.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"wandb: {wandb.__version__}")

torch: 2.10.0+cu128
transformers: 4.51.3
wandb: 0.19.11


## 4. Authentication

### 4a. HuggingFace Token
Required for gated models (Llama-2, etc.). Get your token from [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).

In [10]:
from google.colab import userdata
import os

# Option A: read from Colab secrets (recommended — add HF_TOKEN in the key icon on the left sidebar)
try:
    hf_token = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token
    print("HF_TOKEN loaded from Colab secrets.")
except Exception:
    # Option B: paste directly (less secure)
    import getpass
    hf_token = getpass.getpass("Paste your HuggingFace token: ")
    os.environ['HF_TOKEN'] = hf_token

# Login so transformers picks it up
from huggingface_hub import login
login(token=hf_token, add_to_git_credential=False)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


### 4b. Weights & Biases
The pipeline logs to wandb. You can use **offline mode** (no account needed) or log in.

In [ ]:
import os

# ── Choose ONE of the options below ──────────────────────────────────────────

# Option A: Offline mode (no wandb account needed)
os.environ['WANDB_MODE'] = 'offline'
print("wandb set to offline mode.")

# Option B: Online mode with your wandb account
# import wandb
# wandb.login()  # will prompt for API key
# os.environ['WANDB_SEM_UNC_ENTITY'] = 'your_wandb_username'  # ← update

# ─────────────────────────────────────────────────────────────────────────────

wandb set to offline mode.


## 5. Configure Experiment

Edit these variables to control the run.

In [11]:
# ── Experiment Configuration ──────────────────────────────────────────────────

# Model — small/fast options for Colab free tier:
#   'meta-llama/Llama-3.2-1B-Instruct'  (fast, needs HF token)
#   'TinyLlama/TinyLlama-1.1B-Chat-v1.0' (no token required)
#   'mistralai/Mistral-7B-Instruct-v0.1' (needs ~14 GB VRAM — use A100)
MODEL_NAME = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'

# Dataset: trivia_qa | squad | nq | svamp
DATASET = 'trivia_qa'

# Number of dataset samples (keep small for quick runs)
NUM_SAMPLES = 3

# High-temperature generations per sample (≥3 recommended)
NUM_GENERATIONS = 3

# Random seed
RANDOM_SEED = 42

# Enable Kernel Language Entropy computation
COMPUTE_KLE = True

# Enable EvSemE ablation studies
COMPUTE_ABLATIONS = False

# ─────────────────────────────────────────────────────────────────────────────

print(f"Model:       {MODEL_NAME}")
print(f"Dataset:     {DATASET}")
print(f"Samples:     {NUM_SAMPLES}")
print(f"Generations: {NUM_GENERATIONS}")

Model:       TinyLlama/TinyLlama-1.1B-Chat-v1.0
Dataset:     trivia_qa
Samples:     3
Generations: 3


## 6. Run — Phase 1 & 2: Generate Answers + Compute Uncertainties

This single script runs the full pipeline:
- **Phase 1**: generates answers at low/high temperature, saves `*.pkl` files
- **Phase 2**: computes semantic entropy, KLE, EvSemE, p_true
- **Phase 3**: runs analysis and saves visualizations

In [ ]:
import os
os.chdir('/content/evedential-semantic-entropy')

# Build flags from config variables above
flags = (
    f'--model_name="{MODEL_NAME}"'
    f' --dataset={DATASET}'
    f' --num_samples={NUM_SAMPLES}'
    f' --num_generations={NUM_GENERATIONS}'
    f' --random_seed={RANDOM_SEED}'
)
if COMPUTE_KLE:
    flags += ' --compute_kle'
if COMPUTE_ABLATIONS:
    flags += ' --compute_ablations'

print(f'Running: python semantic_uncertainty/generate_answers.py {flags}')
print('─' * 60)
!python semantic_uncertainty/generate_answers.py {flags}


In [12]:
os.environ['WANDB_MODE'] = 'offline'


In [13]:
!python semantic_uncertainty/generate_answers.py \
    --model_name="meta-llama/Llama-3.2-1B" \
    --dataset=trivia_qa \
    --num_samples=3 \
    --num_generations=3 \
    --random_seed=42



2026-05-11 14:19:57.850942: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.12/dist-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.30.2 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.30.2 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.wa

### Alternative: run via shell (shows live output)

## 7. Inspect Outputs

In [14]:
import glob, os

exp_dir = '/content/evedential-semantic-entropy/EXP'
runs = sorted(glob.glob(f'{exp_dir}/wandb/offline-run-*'), reverse=True)

if runs:
    latest = runs[0]
    print(f"Latest run: {os.path.basename(latest)}")
    print("\nFiles:")
    for f in sorted(glob.glob(f'{latest}/files/*')):
        size = os.path.getsize(f) / 1024
        print(f"  {os.path.basename(f):40s} {size:8.1f} KB")
else:
    print("No runs found in EXP/wandb/")

Latest run: offline-run-20260511_142003-nsuftuqq

Files:
  deberta_entailment_cache.pkl                  0.4 KB
  experiment_details.pkl                        8.5 KB
  files                                         4.0 KB
  output.log                                    0.0 KB
  requirements.txt                             14.4 KB
  train_generations.pkl                        18.0 KB
  uncertainty_measures.pkl                      0.9 KB
  validation_generations.pkl                   57.1 KB
  wandb-metadata.json                           1.2 KB


In [15]:
import pickle
import pandas as pd

latest_files = f'{latest}/files'

# Load uncertainty measures
uq_path = f'{latest_files}/uncertainty_measures.pkl'
if os.path.exists(uq_path):
    with open(uq_path, 'rb') as f:
        uq = pickle.load(f)
    print("uncertainty_measures keys:", list(uq.keys()) if isinstance(uq, dict) else type(uq))
else:
    print(f"Not found: {uq_path}")

uncertainty_measures keys: ['uncertainty_measures', 'semantic_ids', 'graphs', 'validation_is_false', 'validation_unanswerable']


In [16]:
# Load validation generations
gen_path = f'{latest_files}/validation_generations.pkl'
if os.path.exists(gen_path):
    with open(gen_path, 'rb') as f:
        val_gen = pickle.load(f)
    print(f"Validation generations: {len(val_gen)} samples")
    # Show first sample
    sample = val_gen[0]
    print("\nSample keys:", list(sample.keys()) if isinstance(sample, dict) else type(sample))
    if isinstance(sample, dict):
        for k, v in sample.items():
            print(f"  {k}: {str(v)[:120]}")
else:
    print(f"Not found: {gen_path}")

Validation generations: 3 samples


KeyError: 0

## 8. Re-run Uncertainty Computation on Existing Run

If you already have generation outputs and want to recompute uncertainties (skips the expensive generation step).

In [ ]:
# Find the wandb run ID from the latest offline run
if runs:
    run_id = os.path.basename(latest).split('-')[-1]
    print(f"Run ID: {run_id}")

    !cd /content/evedential-semantic-entropy && \
     python semantic_uncertainty/generate_answers.py \
         --skip_generation \
         --eval_wandb_runid={run_id} \
         --compute_kle

## 9. Analyze Results

In [ ]:
if runs:
    run_id = os.path.basename(latest).split('-')[-1]
    !cd /content/evedential-semantic-entropy && \
     python semantic_uncertainty/analyze_results.py \
         --eval_wandb_runid={run_id}

## 10. Download Outputs

Colab sessions expire — download the `EXP/` directory before closing.

In [18]:
# Zip the EXP directory and download it
!zip -r /content/EXP_results.zip /content/evedential-semantic-entropy/EXP/

from google.colab import files
files.download('/content/EXP_results.zip')

updating: content/evedential-semantic-entropy/EXP/ (stored 0%)
updating: content/evedential-semantic-entropy/EXP/experiment_data.txt (deflated 61%)
updating: content/evedential-semantic-entropy/EXP/wandb/ (stored 0%)
updating: content/evedential-semantic-entropy/EXP/wandb/debug.log (deflated 69%)
updating: content/evedential-semantic-entropy/EXP/wandb/debug-internal.log (deflated 72%)
updating: content/evedential-semantic-entropy/EXP/wandb/offline-run-20260511_142003-nsuftuqq/ (stored 0%)
updating: content/evedential-semantic-entropy/EXP/wandb/offline-run-20260511_142003-nsuftuqq/files/ (stored 0%)
updating: content/evedential-semantic-entropy/EXP/wandb/offline-run-20260511_142003-nsuftuqq/files/files/ (stored 0%)
updating: content/evedential-semantic-entropy/EXP/wandb/offline-run-20260511_142003-nsuftuqq/files/files/experiment_details.pkl (deflated 62%)
updating: content/evedential-semantic-entropy/EXP/wandb/offline-run-20260511_142003-nsuftuqq/files/files/validation_generations.pkl (

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 11. Full Experiment (Large Models / More Samples)

For the full paper-scale experiments, use an **A100 GPU** runtime and larger models.

Recommended paper settings:

```bash
python semantic_uncertainty/generate_answers.py \
    --num_samples=500 \
    --model_name=Llama-2-7b-chat \
    --dataset=trivia_qa \
    --num_generations=5 \
    --random_seed=42 \
    --compute_kle \
    --compute_ablations
```

Available models: `Llama-2-7b`, `Llama-2-13b`, `Llama-2-7b-chat`, `Llama-2-13b-chat`, `falcon-7b`, `falcon-40b`, `falcon-40b-instruct`, `falcon-7b-instruct`, `Mistral-7B-v0.1`, `Mistral-7B-Instruct-v0.1`

Available datasets: `trivia_qa`, `squad`, `nq`, `svamp`